In [ ]:
# Multimodal Learning: One Space, Two Views
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part5/20-multimodal.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part5').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part5')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

import copy
from dataclasses import dataclass
from statistics import mean, stdev

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import torch
from torch import Tensor, nn
import torch.nn.functional as F

torch.set_num_threads(6)
torch.set_default_dtype(torch.float32)
torch.manual_seed(6050)

navy, orange, green, wine = "#232D4B", "#E57200", "#2E7D32", "#722F37"

assert _BOOK_ROOT.is_dir()

**Plan**

1. Encode both modalities and normalize them into one comparison space.
2. Score every image against every text at the chosen temperature.
3. Declare the batch diagonal as the supplied pair relation.
4. Average image-to-text and text-to-image cross-entropy.

In [ ]:
def symmetric_contrastive_loss(
    image: Tensor,
    text: Tensor,
    image_encoder: nn.Module,
    text_encoder: nn.Module,
    temperature: float,
) -> Tensor:
    # [1]
    image_embedding = F.normalize(image_encoder(image), dim=-1)
    text_embedding = F.normalize(text_encoder(text), dim=-1)
    # [2]
    logits = image_embedding @ text_embedding.T / temperature
    # [3]
    labels = torch.arange(logits.shape[0], device=logits.device)
    # [4]
    return 0.5 * (
        F.cross_entropy(logits, labels)
        + F.cross_entropy(logits.T, labels)
    )

identity_batch = torch.eye(3)
loss_check = symmetric_contrastive_loss(
    identity_batch, identity_batch, nn.Identity(), nn.Identity(), 0.1
)
assert torch.isfinite(loss_check)

**Plan**

1. Define the reusable helpers: `PairedSplit`, `make_paired_data`, `TwoTower`, `symmetric_loss`, and `pair_loss`.
2. Define the reusable helpers: `retrieval_metrics`, `derangement`, `train_tower`, `report`, and `report_paired_contrast`.
3. Prepare the inputs and fixed settings for the example.
4. Run the five-seed paired-versus-shuffled retrieval study.
5. Report or visualize the measured result.

In [ ]:
# [1]
@dataclass(frozen=True)
class PairedSplit:
    image: Tensor
    text: Tensor


def make_paired_data() -> tuple[PairedSplit, PairedSplit, PairedSplit]:
    generator = torch.Generator().manual_seed(1206050)
    latent_dim, image_dim, text_dim = 5, 14, 11
    latent = torch.randn(1_200, latent_dim, generator=generator)
    image_map = torch.randn(latent_dim, image_dim, generator=generator)
    text_map = torch.randn(latent_dim, text_dim, generator=generator)

    image = torch.tanh(latent @ image_map / latent_dim**0.5)
    text = torch.tanh(latent @ text_map / latent_dim**0.5)
    image += 0.045 * torch.randn(image.shape, generator=generator)
    text += 0.045 * torch.randn(text.shape, generator=generator)

    fit_end, validation_end = 720, 900
    image_mean = image[:fit_end].mean(0, keepdim=True)
    image_std = image[:fit_end].std(0, keepdim=True).clamp_min(1e-6)
    text_mean = text[:fit_end].mean(0, keepdim=True)
    text_std = text[:fit_end].std(0, keepdim=True).clamp_min(1e-6)
    image = (image - image_mean) / image_std
    text = (text - text_mean) / text_std

    return (
        PairedSplit(image[:fit_end], text[:fit_end]),
        PairedSplit(image[fit_end:validation_end], text[fit_end:validation_end]),
        PairedSplit(image[validation_end:], text[validation_end:]),
    )


class TwoTower(nn.Module):
    def __init__(
        self, image_dim: int, text_dim: int,
        width: int = 40, embed_dim: int = 16,
    ):
        super().__init__()
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, width), nn.ReLU(), nn.Linear(width, embed_dim)
        )
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, width), nn.ReLU(), nn.Linear(width, embed_dim)
        )

    def forward(self, image: Tensor, text: Tensor) -> tuple[Tensor, Tensor]:
        image_embedding = F.normalize(self.image_encoder(image), dim=-1)
        text_embedding = F.normalize(self.text_encoder(text), dim=-1)
        return image_embedding, text_embedding


def symmetric_loss(
    image_embedding: Tensor, text_embedding: Tensor, tau: float = 0.08,
) -> Tensor:
    logits = image_embedding @ text_embedding.T / tau  # (B, B)
    labels = torch.arange(logits.shape[0])
    return 0.5 * (
        F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)
    )


@torch.no_grad()
def pair_loss(model: TwoTower, split: PairedSplit) -> float:
    model.eval()
    image_embedding, text_embedding = model(split.image, split.text)
    return float(symmetric_loss(image_embedding, text_embedding))


# [2]
@torch.no_grad()
def retrieval_metrics(model: TwoTower, split: PairedSplit) -> dict[str, float]:
    model.eval()
    image_embedding, text_embedding = model(split.image, split.text)
    scores = image_embedding @ text_embedding.T
    target = torch.arange(scores.shape[0])

    def recall_at_k(matrix: Tensor, k: int) -> float:
        candidates = matrix.topk(k, dim=1).indices
        return float((candidates == target[:, None]).any(dim=1).float().mean())

    return {
        "I→T R@1": recall_at_k(scores, 1),
        "T→I R@1": recall_at_k(scores.T, 1),
        "I→T R@5": recall_at_k(scores, 5),
        "T→I R@5": recall_at_k(scores.T, 5),
    }


def derangement(size: int, generator: torch.Generator) -> Tensor:
    target = torch.arange(size)
    while True:
        order = torch.randperm(size, generator=generator)
        if not bool((order == target).any()):
            return order


def train_tower(
    seed: int,
    fit_split: PairedSplit,
    validation_split: PairedSplit,
    shuffled_pairs: bool,
) -> TwoTower:
    torch.manual_seed(seed)
    model = TwoTower(fit_split.image.shape[1], fit_split.text.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)

    pair_generator = torch.Generator().manual_seed(88_000)
    text_order = (
        derangement(len(fit_split.text), pair_generator)
        if shuffled_pairs else torch.arange(len(fit_split.text))
    )
    schedule_generator = torch.Generator().manual_seed(99_000 + seed)
    schedules = [
        torch.randperm(len(fit_split.image), generator=schedule_generator)
        for _ in range(160)
    ]

    best_validation = float("inf")
    best_state: dict[str, Tensor] | None = None
    for epoch, order in enumerate(schedules, start=1):
        model.train()
        for rows in order.split(120):
            image_embedding, text_embedding = model(
                fit_split.image[rows], fit_split.text[text_order[rows]]
            )
            loss = symmetric_loss(image_embedding, text_embedding)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        if epoch % 5 == 0:
            validation_loss = pair_loss(model, validation_split)
            if validation_loss < best_validation:
                best_validation = validation_loss
                best_state = copy.deepcopy(model.state_dict())

    assert best_state is not None
    model.load_state_dict(best_state)
    return model


# [3]
fit_split, validation_split, held_out_split = make_paired_data()
models: dict[tuple[int, str], TwoTower] = {}
records: list[dict[str, object]] = []

# [4]
for seed in range(6050, 6055):
    for condition, shuffled_pairs in (("paired", False), ("shuffled", True)):
        model = train_tower(seed, fit_split, validation_split, shuffled_pairs)
        models[(seed, condition)] = model
        records.append({
            "seed": seed,
            "condition": condition,
            "validation": retrieval_metrics(model, validation_split),
        })

# The endpoint is consulted after every design and checkpoint decision above.
for record in records:
    model = models[(int(record["seed"]), str(record["condition"]))]
    record["held_out"] = retrieval_metrics(model, held_out_split)


def report(split_name: str, key: str) -> None:
    print(split_name)
    for condition in ("paired", "shuffled"):
        chosen = [record for record in records if record["condition"] == condition]
        fields = list(chosen[0][key].keys())
        summary = []
        for field in fields:
            values = [float(record[key][field]) for record in chosen]
            summary.append(f"{field} {mean(values):.4f} (SD {stdev(values):.4f})")
        print(f"  {condition:8s}: " + "; ".join(summary))


def report_paired_contrast(field: str) -> None:
    differences = []
    for seed in range(6050, 6055):
        paired = next(
            record for record in records
            if record["seed"] == seed and record["condition"] == "paired"
        )
        shuffled = next(
            record for record in records
            if record["seed"] == seed and record["condition"] == "shuffled"
        )
        differences.append(
            float(paired["held_out"][field]) - float(shuffled["held_out"][field])
        )
    print(
        f"paired−shuffled held-out {field}: "
        f"{mean(differences):.4f} (paired SD {stdev(differences):.4f})"
    )


reference_model = models[(6050, "paired")]
parameter_count = sum(p.numel() for p in reference_model.parameters())
# [5]
split_sizes = (
    len(fit_split.image), len(validation_split.image), len(held_out_split.image)
)
print(
    f"split sizes: fit={split_sizes[0]}, validation={split_sizes[1]}, "
    f"held-out={split_sizes[2]}"
)
print(f"parameters: {parameter_count:,}; model seeds: 6050–6054")
report("validation", "validation")
report("held-out endpoint", "held_out")
report_paired_contrast("I→T R@1")
report_paired_contrast("T→I R@1")
print(
    f"held-out random-ranking expectations: "
    f"R@1={1 / 300:.4f}; R@5={5 / 300:.4f}"
)

**Plan**

1. Define the reusable helpers: `held_values` and `score_window`.

In [ ]:
# [1]
def held_values(condition: str, field: str) -> list[float]:
    return [
        float(record["held_out"][field])
        for record in records
        if record["condition"] == condition
    ]


@torch.no_grad()
def score_window(model: TwoTower, split: PairedSplit, size: int = 40) -> Tensor:
    model.eval()
    image_embedding, text_embedding = model(split.image[:size], split.text[:size])
    return image_embedding @ text_embedding.T

**Plan**

1. Define the reusable `two_dim_view` helper.
2. Prepare the inputs and fixed settings for the example.

In [ ]:
# [1]
def two_dim_view(model: TwoTower, count: int = 20) -> tuple[Tensor, Tensor]:
    model.eval()
    with torch.no_grad():
        image_embedding, text_embedding = model(
            held_out_split.image[:count], held_out_split.text[:count]
        )
    both = torch.cat([image_embedding, text_embedding])
    centered = both - both.mean(0)
    _, _, right = torch.linalg.svd(centered, full_matrices=False)
    projected = centered @ right[:2].T
    return projected[:count], projected[count:]

# [2]
torch.manual_seed(6050)
untrained = TwoTower(held_out_split.image.shape[1], held_out_split.text.shape[1])
views = [(untrained, "before training:\nsame width, unrelated coordinates"),
         (models[(6050, "paired")], "after paired training:\nmates land together")]